# Phase 11.7 — Recommendation Evaluation

Offline curated-judgment evaluation (not user-behavior ground truth).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.recommendation.evaluation import (
    RecommendationEvaluationConfig,
    RecommendationSeedPipelineSnapshot,
    evaluate_recommendation_benchmark,
    evaluation_result_to_dict,
    load_recommendation_benchmark,
)
from productiq.recommendation.contracts import RankedRecommendation, RecommendationCandidate, RecommendationCandidateSource

benchmark = load_recommendation_benchmark(
    ROOT / "resources" / "evaluation" / "productiq_recommendation_v1.json"
)
benchmark.metadata.benchmark_name, len(benchmark.cases)

In [ ]:
# Example: attach observed pipeline outputs per seed (here: illustrative placeholders).
# Production runs should populate snapshots from 11.2→11.6 execution without changing scores.

def _ranked(product_id: str, rank: int, score: float) -> RankedRecommendation:
    return RankedRecommendation(
        product_id=product_id,
        rank=rank,
        recommendation_score=score,
        candidate=RecommendationCandidate(
            product_id=product_id,
            sources=(RecommendationCandidateSource.VECTOR,),
            candidate_generation_score=score,
        ),
    )


case = benchmark.cases[0]
judged_ids = [row.product_id for row in case.relevance_judgments]
snapshot = RecommendationSeedPipelineSnapshot(
    seed_product_id=case.seed_product_id,
    requested_top_k=5,
    candidate_pool_product_ids=tuple(judged_ids),
    ranked_recommendations=tuple(
        _ranked(pid, idx + 1, 1.0 - idx * 0.05) for idx, pid in enumerate(judged_ids[:3])
    ),
    final_recommendations=tuple(
        _ranked(pid, idx + 1, 1.0 - idx * 0.05) for idx, pid in enumerate(judged_ids[:2])
    ),
)
case.seed_product_id, case.relevance_judgments[:2], snapshot

In [ ]:
config = RecommendationEvaluationConfig(k_values=(1, 5), evaluation_top_k=5, min_relevant_grade=2)
# Full benchmark evaluation requires a snapshot for every seed.
snapshots = {case.seed_product_id: snapshot for case in benchmark.cases}
snapshots[case.seed_product_id] = snapshot
result = evaluate_recommendation_benchmark(benchmark, snapshots, config=config)
evaluation_result_to_dict(result)["comparison"]